In [1]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion, Formula
import rpy2.robjects.packages as rpackages
from rpy2.robjects.packages import importr

In [3]:
import os
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype

In [5]:
from deconveil.utils_fit import *
from deconveil.utils_processing import *

#### Load data

In [7]:
data_path = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/data/deconveil_test/rna.csv"
rna = pd.read_csv(data_path, index_col=0)

In [9]:
rna.head()

,CRC.SW.U0001.T,CRC.SW.U0002.T,CRC.SW.U0004.T,CRC.SW.U0030.T,CRC.SW.U0066.T,CRC.SW.U0067.T,CRC.SW.U0069.T,CRC.SW.U0070.T,CRC.SW.U0072.T,CRC.SW.U0073.T,...,CRC.SW.UM164.T,CRC.SW.UM165.T,CRC.SW.UM166.T,CRC.SW.UM167.T,CRC.SW.UM168.T,CRC.SW.UM169.T,CRC.SW.UM170.T,CRC.SW.UM171.T,CRC.SW.UM172.T,CRC.SW.UM173.T
A1BG,12,3,14,4,4,6,6,8,7,4,...,2,0,2,2,0,0,0,0,0,7
A1CF,1899,2448,2353,2875,1065,4168,150,2111,655,2438,...,2911,2974,153,1958,992,2017,1883,2097,1124,253
A2M,43569,16953,12739,14427,16697,16729,28844,16593,19598,6649,...,4392,15168,4399,3894,4908,7267,6542,1952,8560,9687
A2ML1,1,3,1,4,3,0,1,10,0,0,...,10,0,0,1,17,2,2,11,8,0
A3GALT2,0,0,0,2,1,0,0,1,2,0,...,0,0,0,2,0,0,0,2,0,0


In [11]:
data_path = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/data/deconveil_test/metadata.csv"
meta = pd.read_csv(data_path, index_col=0)
meta.head()

,MSIstatus,CNstate
CRC.SW.U0001.T,MSS,Loss
CRC.SW.U0002.T,MSS,Loss
CRC.SW.U0004.T,MSS,Loss
CRC.SW.U0030.T,MSS,Neutral
CRC.SW.U0066.T,MSS,Loss


In [13]:
# Define categorical order (reference = first level)
msi_order = CategoricalDtype(categories=["MSI", "MSS"], ordered=True)
cn_order  = CategoricalDtype(categories=["Neutral", "Loss", "Gain"], ordered=True)

# Apply to the metadata DataFrame
meta["MSIstatus"] = meta["MSIstatus"].astype(msi_order)
meta["CNstate"]   = meta["CNstate"].astype(cn_order)

In [ ]:
#### Model test

In [17]:
def run_deseq2_multifactor(
    rna_counts,
    metadata,
    design_formula="~ condition * CN",
    #shrink_coef=None,
    output_dir=None,
    shrink_type="apeglm"
):
    """
    Run DESeq2 multifactor model using only RNA counts and metadata.

    This version assumes any CN-related information (e.g. CNmean, CNstatus)
    is already present in the metadata table.

    Parameters
    ----------
    rna_counts : pd.DataFrame
        Gene expression count matrix (genes x samples)
    metadata : pd.DataFrame
        Sample-level metadata (samples x covariates, must already include CN or CNmean)
    design_formula : str
        R-style formula for DESeq2 (e.g. "~ condition + CN" or "~ condition * CN")
    shrink_coef : str, optional
        Coefficient name for LFC shrinkage (must match resultsNames(dds))
    shrink_type : str, optional
        Shrinkage estimator type ("apeglm" or "normal", etc.)

    Returns
    -------
    dds : rpy2.robjects.RObject
        DESeq2 dataset object (for further R operations)
    res_df : pd.DataFrame
        DESeq2 results table
    """

    deseq2 = importr("DESeq2")

    # Align samples between counts and metadata
    shared_samples = rna_counts.columns.intersection(metadata.index)
    if len(shared_samples) == 0:
        raise ValueError("No overlapping samples found between RNA and metadata.")

    rna_counts = rna_counts[shared_samples]
    metadata = metadata.loc[shared_samples]

    # Ensure counts have genes as rows, samples as columns
    if rna_counts.shape[0] < rna_counts.shape[1]:
        print("Transposing RNA count matrix (genes as rows).")
        rna_counts = rna_counts.T

    # Convert categorical variables
    cat_cols = metadata.select_dtypes(include=["object", "category"]).columns
    for col in cat_cols:
        metadata[col] = metadata[col].astype("category")

    # Prepare row/column names
    rna_counts.index = rna_counts.index.astype(str)
    metadata.index = metadata.index.astype(str)

    # Convert to R objects
    with conversion.localconverter(ro.default_converter + pandas2ri.converter):
        rna_counts_r = conversion.py2rpy(rna_counts.astype(int))
        metadata_r = conversion.py2rpy(metadata)

    # Assign to R environment
    ro.globalenv["rna_counts_r"] = rna_counts_r
    ro.globalenv["metadata_r"] = metadata_r
    ro.globalenv["gene_names"] = ro.StrVector(rna_counts.index.tolist())
    ro.globalenv["sample_names"] = ro.StrVector(metadata.index.tolist())
    ro.r("rownames(rna_counts_r) <- gene_names")
    ro.r("rownames(metadata_r) <- sample_names")

    # Ensure factors are not ordered
    ro.r('''
    for (col in colnames(metadata_r)) {
      if (is.factor(metadata_r[[col]]) && is.ordered(metadata_r[[col]])) {
        metadata_r[[col]] <- factor(metadata_r[[col]], ordered = FALSE)
      }
    }
    ''')

    # Check alignment
    aligned = ro.r('all(colnames(rna_counts_r) == rownames(metadata_r))')[0]
    if not aligned:
        raise ValueError("Sample alignment mismatch between rna_counts and metadata.")

    # Run DESeq2
    print(f"Running DESeq2 with design: {design_formula}")
    dds = deseq2.DESeqDataSetFromMatrix(
        countData=ro.globalenv["rna_counts_r"],
        colData=ro.globalenv["metadata_r"],
        design=Formula(design_formula)
    )

    dds = deseq2.DESeq(dds)
    ro.globalenv["dds"] = dds

    # Get coefficients
    coef_names = list(ro.r("resultsNames(dds)"))
    coef_names = [c for c in coef_names if c.lower() != "intercept"]
    print("\nAvailable coefficients:")
    for c in coef_names:
        print("  -", c)

    # Prepare output folder
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    results_all = {}

    # Perform shrinkage for all coefficients
    for coef in coef_names:
        print(f"\nPerforming LFC shrinkage for: {coef}")
        try:
            res_shr = deseq2.lfcShrink(dds, coef=coef, type=shrink_type)
        except Exception:
            print("↩️ Retrying with shrink_type='normal' ...")
            res_shr = deseq2.lfcShrink(dds, coef=coef, type="normal")

        with conversion.localconverter(ro.default_converter + pandas2ri.converter):
            res_df = conversion.rpy2py(ro.r("as.data.frame")(res_shr))

        results_all[coef] = res_df

        if output_dir:
            csv_path = os.path.join(
                output_dir, f"{coef}.csv".replace(":", "_").replace("/", "_")
            )
            res_df.to_csv(csv_path)
            print(f"✅ Saved {csv_path}")

    # Combine into one long table
    combined = pd.concat(
        [df.assign(contrast=coef) for coef, df in results_all.items()],
        axis=0
    )
    if output_dir:
        combined_path = os.path.join(output_dir, "deseq2_all_shrinkage_results.csv")
        combined.to_csv(combined_path)
        print(f"\n📁 Combined results saved to: {combined_path}")

    return dds, results_all, combined

In [19]:
res_df, dds = run_deseq2_multifactor(
    rna_counts=rna,
    metadata=meta,
    design_formula="~MSIstatus * CNstate",
    shrink_type="apeglm",
    output_dir = "/Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/results/"
)

Running DESeq2 with design: ~MSIstatus * CNstate


R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 1025 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  



Available coefficients:
  - MSIstatus_MSS_vs_MSI
  - CNstate_Loss_vs_Neutral
  - CNstate_Gain_vs_Neutral
  - MSIstatusMSS.CNstateLoss
  - MSIstatusMSS.CNstateGain

Performing LFC shrinkage for: MSIstatus_MSS_vs_MSI


R callback write-console: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
  
R callback write-console: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
  


✅ Saved /Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/results/MSIstatus_MSS_vs_MSI.csv

Performing LFC shrinkage for: CNstate_Loss_vs_Neutral


...

R callback write-console: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
  


✅ Saved /Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/results/CNstate_Gain_vs_Neutral.csv

Performing LFC shrinkage for: MSIstatusMSS.CNstateLoss
✅ Saved /Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/results/MSIstatusMSS.CNstateLoss.csv

Performing LFC shrinkage for: MSIstatusMSS.CNstateGain


R callback write-console: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
  


✅ Saved /Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/results/MSIstatusMSS.CNstateGain.csv

📁 Combined results saved to: /Users/katsiarynadavydzenka/Documents/PhD_AI/CRC/results/deseq2_all_shrinkage_results.csv


ValueError: too many values to unpack (expected 2)